In [1]:
from langchain_core.messages import SystemMessage, HumanMessage
from langchain_google_genai import ChatGoogleGenerativeAI
from dotenv import load_dotenv
import os

In [2]:
load_dotenv()
project = os.getenv('GOOGLE_CLOUD_PROJECT')


# model
llm = ChatGoogleGenerativeAI(
    model = "gemini-2.5-flash-lite",
    vertexai = True,
    project = project
)

In [3]:
from langchain.tools import tool

@tool
def say_hello(name:str) -> str:
    """This function is used to greet a person

    Args:
        name (str): name

    Returns:
        str: Greeting
    """
    return f"Hello {name}"


In [4]:
llm_with_tools = llm.bind_tools([say_hello])

In [5]:
# llm which is not aware of tools
response = llm.invoke("Greet shyam")
response

AIMessage(content='Hello Shyam!', additional_kwargs={}, response_metadata={'finish_reason': 'STOP', 'model_name': 'gemini-2.5-flash-lite', 'safety_ratings': [], 'model_provider': 'google_genai'}, id='lc_run--019d5843-a9c4-74e0-81eb-31433d6d7001-0', tool_calls=[], invalid_tool_calls=[], usage_metadata={'input_tokens': 3, 'output_tokens': 3, 'total_tokens': 6, 'input_token_details': {'cache_read': 0}})

In [6]:
response = llm_with_tools.invoke("Greet shyam")
response

AIMessage(content='', additional_kwargs={'function_call': {'name': 'say_hello', 'arguments': '{"name": "shyam"}'}}, response_metadata={'finish_reason': 'STOP', 'model_name': 'gemini-2.5-flash-lite', 'safety_ratings': [], 'model_provider': 'google_genai'}, id='lc_run--019d5843-b8f4-7ce2-9254-ff5fb7faadd2-0', tool_calls=[{'name': 'say_hello', 'args': {'name': 'shyam'}, 'id': '6232b706-9b27-4f50-8fb0-77b7bf9c1639', 'type': 'tool_call'}], invalid_tool_calls=[], usage_metadata={'input_tokens': 38, 'output_tokens': 6, 'total_tokens': 44, 'input_token_details': {'cache_read': 0}})

In [7]:
from langchain.agents import create_agent
# create a agent
agent = create_agent(
    model=llm,
    tools=[say_hello]
)

In [8]:
agent_response = agent.invoke({
    "messages": [
        ("user", "Greet shyam")
    ]
})

In [9]:
agent_response['messages'][-1].pretty_print()

================================== Ai Message ==================================

Hello shyam


In [10]:
agent_response['messages']

[HumanMessage(content='Greet shyam', additional_kwargs={}, response_metadata={}, id='ced94d95-53fe-4fa3-8469-6679d1064e43'),
 AIMessage(content='', additional_kwargs={'function_call': {'name': 'say_hello', 'arguments': '{"name": "shyam"}'}}, response_metadata={'finish_reason': 'STOP', 'model_name': 'gemini-2.5-flash-lite', 'safety_ratings': [], 'model_provider': 'google_genai'}, id='lc_run--019d5843-bc4b-7b20-ad2d-d614fdffe27b-0', tool_calls=[{'name': 'say_hello', 'args': {'name': 'shyam'}, 'id': '8c1be77b-e264-4a26-ab67-361dae0818d3', 'type': 'tool_call'}], invalid_tool_calls=[], usage_metadata={'input_tokens': 38, 'output_tokens': 6, 'total_tokens': 44, 'input_token_details': {'cache_read': 0}}),
 ToolMessage(content='Hello shyam', name='say_hello', id='12b96885-fb71-49d3-a536-c504de1371a1', tool_call_id='8c1be77b-e264-4a26-ab67-361dae0818d3'),
 AIMessage(content='Hello shyam', additional_kwargs={}, response_metadata={'finish_reason': 'STOP', 'model_name': 'gemini-2.5-flash-lite', 's

In [15]:
## I want to search internet
load_dotenv()
#os.getenv('TAVILY_API_KEY')


True

In [16]:
!uv add langchain-tavily

Resolved 85 packages in 1ms
Checked 81 packages in 2ms


In [17]:
# lets create tavily search tool
from langchain_tavily import TavilySearch

tavily_search_tool = TavilySearch(
    max_results = 3,
    topic = "news"
)

In [18]:
# lets create an agent with tavily search tool

agent = create_agent(
    model=llm,
    tools=[tavily_search_tool]
)

In [19]:
response = agent.invoke({
    "messages": "Get me latest news about gpu's "
})

In [20]:
response['messages']

[HumanMessage(content="Get me latest news about gpu's ", additional_kwargs={}, response_metadata={}, id='c432f548-0d4f-4798-ad15-cc4bb91f14f1'),
 AIMessage(content='', additional_kwargs={'function_call': {'name': 'tavily_search', 'arguments': '{"topic": "news", "query": "latest news about gpus"}'}}, response_metadata={'finish_reason': 'STOP', 'model_name': 'gemini-2.5-flash-lite', 'safety_ratings': [], 'model_provider': 'google_genai'}, id='lc_run--019d584d-2ae4-7452-9a30-227000b4e018-0', tool_calls=[{'name': 'tavily_search', 'args': {'topic': 'news', 'query': 'latest news about gpus'}, 'id': 'a726a78a-eaa4-46df-a508-3f505fd126cc', 'type': 'tool_call'}], invalid_tool_calls=[], usage_metadata={'input_tokens': 1298, 'output_tokens': 13, 'total_tokens': 1311, 'input_token_details': {'cache_read': 0}}),
 ToolMessage(content='{"query": "latest news about gpus", "follow_up_questions": null, "answer": null, "images": [], "results": [{"url": "https://videocardz.com/newz/new-rowhammer-attacks-t

In [21]:
response['messages'][-1].pretty_print()

================================== Ai Message ==================================

Here are some of the latest news about GPUs:

*   **New Rowhammer attacks target modern GPUs:** The GeForce RTX 3060 and RTX A6000 are confirmed to be vulnerable to these attacks.
*   **Nvidia H100 GPU prices surge:** Rental prices for these GPUs have increased by approximately 40% due to high demand.
*   **Unusual cooling mod for GeForce RTX 3080:** A YouTuber has tested a workstation AIO cooler on the RTX 3080, significantly reducing VRAM temperatures.


In [22]:
@tool
def find_friends(name:str) -> list:
    """Find friends of a person 
    """
    return ["Amar", "Akbar", "Anthony"]

In [23]:
agent = create_agent(
    model=llm,
    tools = [find_friends]
)

In [24]:
response = agent.invoke({
    "messages": "Find friends of Ram"
})

In [25]:
response['messages'][-1].pretty_print()

================================== Ai Message ==================================

Here are Ram's friends: Amar, Akbar, and Anthony.


In [26]:
response['messages']

[HumanMessage(content='Find friends of Ram', additional_kwargs={}, response_metadata={}, id='62d50d22-49f7-447b-86f9-08727ed6e8d7'),
 AIMessage(content='', additional_kwargs={'function_call': {'name': 'find_friends', 'arguments': '{"name": "Ram"}'}}, response_metadata={'finish_reason': 'STOP', 'model_name': 'gemini-2.5-flash-lite', 'safety_ratings': [], 'model_provider': 'google_genai'}, id='lc_run--019d584d-8cd9-70d0-87da-3ac3bbc3d392-0', tool_calls=[{'name': 'find_friends', 'args': {'name': 'Ram'}, 'id': 'e6401717-d0f8-49ad-88bc-f63630997e73', 'type': 'tool_call'}], invalid_tool_calls=[], usage_metadata={'input_tokens': 16, 'output_tokens': 5, 'total_tokens': 21, 'input_token_details': {'cache_read': 0}}),
 ToolMessage(content='["Amar", "Akbar", "Anthony"]', name='find_friends', id='90a3d49e-1331-4932-a408-d6962bba5952', tool_call_id='e6401717-d0f8-49ad-88bc-f63630997e73'),
 AIMessage(content="Here are Ram's friends: Amar, Akbar, and Anthony.", additional_kwargs={}, response_metadata

In [27]:
agent = create_agent(
    model=llm,
    tools = [find_friends, say_hello]
)

In [28]:
response = agent.invoke({
    "messages": "Find friends of Ram and ensure you greet each friend"
})

In [29]:
for message in response['messages']:
    message.pretty_print()

================================ Human Message =================================

Find friends of Ram and ensure you greet each friend
================================== Ai Message ==================================
Tool Calls:
  find_friends (157657f2-3cf1-44a2-8d2b-c1bac377f355)
 Call ID: 157657f2-3cf1-44a2-8d2b-c1bac377f355
  Args:
    name: Ram
================================= Tool Message =================================
Name: find_friends

["Amar", "Akbar", "Anthony"]
================================== Ai Message ==================================
Tool Calls:
  say_hello (5a66cd9b-0c07-456b-b30b-753f285f1557)
 Call ID: 5a66cd9b-0c07-456b-b30b-753f285f1557
  Args:
    name: Amar
  say_hello (9d65a8c9-0b95-45ea-b257-01bf64c57871)
 Call ID: 9d65a8c9-0b95-45ea-b257-01bf64c57871
  Args:
    name: Akbar
  say_hello (197aeef6-e43c-42d5-bb39-a7fdbd88d140)
 Call ID: 197aeef6-e43c-42d5-bb39-a7fdbd88d140
  Args:
    name: Anthony
================================= Tool Message ============